# ZTE — ZuCo Thought Embedding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/zte_colab_v2.ipynb)

**Reading EEG back into language, honestly.** This notebook is the operating manual for the ZTE pipeline: an
**encoder** that maps word- and sentence-level EEG into a shared embedding space, and a **decoder** that reads text
out of it through a frozen language model on a small trainable leash.

The north star is clinical — helping people with ALS, paralysis and locked-in syndrome turn EEG into natural
language. Work aimed at a vulnerable population has to earn trust, so **an overclaimed result is worse than no
result**. Everything below is built around that: bootstrap confidence intervals over point estimates, permutation
nulls, held-out splits that hold out whole *people*, provenance travelling with every artifact, and null results
reported plainly.

---

## What this notebook is for

| You want to | Go to |
| --- | --- |
| Set up a fresh Colab runtime | §1 – §3 |
| Mount Drive and prepare the data once | §4 – §5 |
| Understand what the current encoder can and cannot do | §6 |
| Train the encoder (one arm, the ablations, or the full sweep) | §7 |
| Train and audit the decoder | §8 |
| Run the entire study with one resumable command | §9 |
| Look at everything you have ever run, visually | §10 |
| Persist, resume and continue offline | §11 – §12 |

## How to read a number in this project

Four rules, each of which has already caught a wrong conclusion here.

1. **`held_out_retrieval` is the result. `sentence_retrieval` is not.** The pooled number is computed over the
   training subjects as well as the held-out one, so it rewards memorising the brains you have rather than reaching
   the one you do not. It inverted the champion once already.
2. **Top-1 on 700 queries expects exactly one hit by chance.** A "0.006 vs 0.001" headline is three hits at
   $p \approx 0.08$. Read **rank percentile** with its confidence interval, and read Top-K as *hit counts* with an
   exact binomial tail.
3. **Sentence length is 5.14 of the 9.45 bits.** ZuCo segments words by eye tracking, so the model gets the word
   count for free, and a length-only oracle beats every encoder measured here on every top-k. Any retrieval number
   quoted without saying whether it is length-stratified is not a claim.
4. **Generation is not a headline unless the verdict says so.** `verdict['generation_above_controls']` ANDs over an
   honest split, no candidate set, every pre-registered control beaten, a permutation $p < 0.05$, and a
   prefix-influence KL above the floor. A control that did not run *fails* its clause.

> **A synthetic run is never a result.** `--synthetic` exists to prove the plumbing works. Every number that is
> allowed to leave this notebook came from real ZuCo.

## 1 · Provision the runtime

Colab ships an older Python than ZTE requires (`>=3.14`), so `uv` provisions the pinned interpreter and installs
everything into a cached virtualenv. The clone is shallow and hard-resets to `main`, so only git-tracked files move —
your Drive folder and any local cache are untouched.

Re-running this cell on a warm runtime is fast and idempotent.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

Colab's kernel is an **older interpreter than the 3.14 ZTE requires**, so `import zte` here is a `SyntaxError`. It
never needs to: every capability arrives through `zte-colab`, one subcommand per question, each printing a single
JSON object on stdout with its logs on stderr. The kernel's job is to render those payloads.

`colab()` below is that bridge, and it is the only way this notebook reaches the package. Everything else in a code
cell is the standard library, `IPython.display`, `google.colab`, and Colab's own `pandas` / `plotly`.

The environment it applies fixes four failures that are silent rather than loud, which is why they are set here and
not left to chance --- every `!` command below inherits the kernel's environment.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) — Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded — authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} · zte {ENV["venv"]["zte"]}   ← every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   ← this cell; renders payloads, never imports zte')
print(f'env    : {", ".join(ENV["env"])}')

## 3 · What hardware did you get, and what will ZTE do with it

ZTE resolves one device spec and every component reads it — precision, DataLoader workers, pinned memory, static
shapes on TPU. The cell prints both what Colab gave you and what ZTE will do with it, so an out-of-memory kill later
is predictable rather than a mystery. It re-reads the payload, so change the runtime type and re-run just this cell.

**A raw-conformer batch turns every (sentence, word) pair into its own 105×350 attention problem.** That is tens of
gigabytes of activations on a full batch, which is why `model.grad_checkpoint: true` is set in every raw config —
numerically identical gradients, roughly 30% slower, and the difference between fitting on a 16 GB GPU and not.

In [ ]:
ENV = colab('env')
PLAN, ACCEL, RES = ENV['plan'], ENV['accelerator'], ENV['resources']

print(f'accelerator : {ACCEL["name"]}   (torch {ACCEL["torch_version"]})')
print(f'ZTE backend : {PLAN["backend"]} · {PLAN["device"]}')
print(f'precision   : {PLAN["autocast_dtype"]}   (mixed precision {"on" if PLAN["mixed_precision"] else "off"})')
print(f'dataloader  : {PLAN["dataloader_workers_auto"]} workers · pin_memory {PLAN["pin_memory"]}')
print(f'machine     : {RES["ram_gb"]} GB RAM · {RES["cpu_count"]} cores · {RES["free_disk_gb"]} GB free disk')
print(f'GPU         : {RES["gpu"]["name"]} ({RES["gpu"]["total_gb"]} GB)' if RES['gpu'] else 'GPU         : none')

if not RES['gpu'] and PLAN['backend'] == 'cpu':
    print('\n⚠  No accelerator. Runtime → Change runtime type → GPU before training anything real.')
if RES['low_ram']:
    print(
        f'\n⚠  Raw-EEG bundles are ~24 GB materialised and this VM has under {RES["low_ram_threshold_gb"]} GB.'
        ' Prefer Runtime → Change runtime type → High-RAM.'
    )

## 4 · Drive is the workspace

**Everything durable lives on Drive.** A Colab VM can vanish without warning; a multi-hour sweep whose only copy was
on the VM disk is a multi-hour sweep you get to run again.

The layout is one shared folder for data plus one dated folder per session:

```text
Sharables/ZTE/
├── prepared/                     # cached feature bundles, NOT date-stamped: built once, reused forever
├── ZuCo Dataset/                 # the raw .mat files
└── YYYY-MM-DD/                   # one folder per session
    ├── experiments/              # every run: config, checkpoints, evaluation, figures
    ├── analysis/                 # the study dashboard and its tidy tables
    └── archives/                 # provenance-stamped zips
```

**Where each kind of work writes.** Training checkpoints go to the VM's fast local disk and are mirrored to Drive
after every stage, because a Drive FUSE stall mid-`torch.save` is a torn checkpoint. Everything else — evaluation,
generation, the analysis dashboard, the archives — is written straight to Drive. Set `WRITE_MODE = 'drive'` below to
put checkpoints on Drive too; it is slower and less robust, and it survives a VM reset without the mirror step.

**To resume an interrupted session**, set `RESUME_DATE` to that session's folder name. Everything then points at the
same place and every `--resume` finds its work already done.

### What is safe at every moment

The rule is that nothing expensive is ever more than one epoch, or one stage, away from durable storage.

| written | when | why then |
| --- | --- | --- |
| `best.pt` | the moment it improves | it is the result; losing it loses the run's whole point |
| `last.pt` | every epoch | it is what `--resume` reads, so a reclaimed VM costs one epoch |
| `ckpt_epoch*.pt` | never mirrored | rotation history: `keep_last` extra copies of a large file that a fresh VM cannot use |
| the run directory | each stage | config, `history.json`, evaluation, figures, TensorBoard |
| evaluation · generation · analysis · studio | straight to the durable root | expensive, and none of it resumes -- recomputing is the only recovery |
| the prepared feature bundle | once, ever | content-addressed and *not* date-stamped, so every future session reuses it |

A mirror that fails must never kill a run, so failures are logged and training continues -- but a mirror that has
been silently failing for forty epochs is worse than one that failed loudly, so consecutive failures escalate to an
error naming the missing file.

Restoring a run directory from Drive gives you `best.pt` and `last.pt` and no rotation history. That is deliberate,
and `--resume` handles it: it tries `last.pt`, then any epoch files, then `best.pt`. Even a `last.pt` torn by the
write that was in flight when the machine went away costs the epochs since the last improvement, not the run.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# Set to an existing folder name (e.g. '2026-08-13') to resume that session; None starts today's.
RESUME_DATE: str | None = None
# 'local+mirror' trains on the VM disk and copies to Drive after each stage (recommended).
# 'drive' writes runs straight to Drive: slower, but nothing to mirror if the VM dies mid-epoch.
WRITE_MODE: str = 'local+mirror'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)

# Every `!` command below inherits these, so the bundle cache, the data root and the backup target are wired once.
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_DIR: str = SESSION['session_dir']
DRIVE_RUNS: str = SESSION['drive_runs']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
LOCAL_RUNS: str = SESSION['local_runs']
OUT_ROOT: str = SESSION['out_root']
DRIVE_BACKUP: str = SESSION['drive_backup']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "new"})')
print(f'Drive     : {SESSION["drive_root"]}   (mounted: {SESSION["drive_mounted"]})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}   (backed up to {DRIVE_BACKUP})')
print(f'analysis  : {DRIVE_ANALYSIS}')
print(f'prepared  : {PREPARED_DRIVE}   (staged on the VM at {PREPARED_LOCAL})')

### 4a · Helpers this notebook uses everywhere

Small, boring and worth reading once: where runs can be found, which checkpoint a name resolves to, and how to move
state between the VM and Drive in both directions. Each one is a thin renderer over a `zte-colab` payload, so the
search order and the exclusion rules live in the package and are tested there rather than drifting in a notebook.

In [ ]:
import pathlib
import shutil


def find_runs(*extra: str, headline: bool = False) -> list[dict[str, Any]]:
    """Every run reachable right now — each dated Drive session newest first, then the local disk.

    A run is anything with a `config.yaml`, so one a reclaimed VM killed mid-training is still listed; `evaluated`
    is what says whether it got as far as producing numbers.
    """
    roots = [*extra, LOCAL_RUNS]
    flags = ['--headline'] if headline else []

    return colab('runs', '--drive', ZTE_DRIVE, '--experiments', *roots, *flags)['runs']


def every_session() -> list[str]:
    """Every dated session's run folder on Drive, newest first — what the analysis section reads across."""
    return colab('runs', '--drive', ZTE_DRIVE)['sessions']


def resolve_ckpt(run_name: str, which: str = 'best') -> str:
    """Finds a run's checkpoint, Drive first, so a fresh VM can decode a session it did not train.

    A missing `best.pt` never falls back to `last.pt`: they are different models, and swapping them silently
    misattributes the number.
    """
    for run in colab('runs', '--drive', ZTE_DRIVE, '--experiments', LOCAL_RUNS, '--run', run_name)['runs']:
        if path := run['checkpoints'][which]:
            print(f'{which}.pt for {run_name}: {"Drive" if run["source"] == "drive" else "local disk"}\n  {path}')
            return path

    raise FileNotFoundError(f'no {which}.pt for {run_name!r} on Drive or locally; train it first (Section 7).')


def durable(*parts: str) -> str:
    """A path under the durable root: this session's Drive folder on Colab, `res/` on a machine without it.

    Everything expensive that does *not* resume -- the analysis dashboard, the studio page, the rebaseline audit --
    is written here rather than written locally and copied later, so a VM reclaimed during the *next* cell cannot
    take it.
    """
    root = DRIVE_DIR if SESSION['drive_mounted'] else 'res'
    path = os.path.join(root, *parts)
    os.makedirs(os.path.dirname(path) or path, exist_ok=True)

    return path


def _mirror(direction: str, date: str, sub: str, local: str | None) -> None:
    """Runs one mirror and reports what moved, or why nothing did."""
    where = ['--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *(('--local', local) if local else ())]
    payload = colab('mirror', *where, '--direction', direction, '--date', date, '--sub', sub)

    if reason := payload['skipped_reason']:
        print(f'nothing mirrored: {reason}')
        return

    print(f'{payload["src"]} -> {payload["dst"]}   ({payload["copied"]} copied, {payload["failed"]} failed)')


def mirror_to_drive(local: str | None = None, sub: str = 'experiments') -> None:
    """Copy the VM's runs to Drive, minus what is rebuildable, so the session survives the machine."""
    _mirror('up', RUN_DATE, sub, local)


def restore_from_drive(run_date: str | None = None, sub: str = 'experiments', local: str | None = None) -> None:
    """Pull a session's runs back to the VM so every `--resume` finds its work after a runtime reset."""
    _mirror('down', run_date or RUN_DATE, sub, local)


def show_resources() -> None:
    """Prints RAM / GPU / disk as they stand now, so an out-of-memory kill is predictable rather than a mystery."""
    res = colab('env')['resources']
    gpu = f'{res["gpu"]["name"]} ({res["gpu"]["total_gb"]} GB)' if res['gpu'] else 'none'
    print(f'RAM {res["ram_gb"]} GB · {res["cpu_count"]} cores · {res["free_disk_gb"]} GB free disk · GPU {gpu}')


show_resources()

## 5 · Prepare the data once, on Drive, and never again

`zte-prepare` keys every shipped config by a hash of the fields that actually change the processed bundle, asks the
persistent Drive store what it already holds, and builds only what is genuinely absent. A fully-prepared project
never touches the raw `.mat` files again.

Two paths, and the split matters. Both were exported by the session cell above, so nothing here is set by hand:

- **`ZTE_CACHE_REMOTE`** → `Sharables/ZTE/prepared`. Persistent and *not* date-stamped, because a feature bundle is a
  property of the data and the config, not of the session that happened to build it.
- **`DATA_CACHE`** → a local copy on the VM's fast disk. Bundles are content-addressed and immutable, so staging is a
  copy-if-absent and never a re-computation.

Building the raw bundles from scratch is the single longest step in the project. Once it is on Drive it is done
forever, for every future session.

In [ ]:
!uv run zte-prepare --root "{DATA_DIR}" --configs --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

### 5a · Is the signal even there? — the model-free audit

Before any model, `zte-audit` measures the associations in the data itself. Run it once per dataset and read it
before believing any result. Two findings from it govern every design decision below:

- **Task is an alias for the stimulus.** Cramér's $V(\text{task}, \text{stimulus}) = 0.998$ and *no* sentence appears
  under both tasks, so a task-invariance loss also deletes content. The answer is to match negatives within task, not
  to turn the adversary up.
- **Word length and word frequency are the same variable here** ($|\rho| = 0.99$), and eye-tracking measures carry
  subject identity ($\eta \approx 0.25$).

Running it again is free: the report is skipped when it was already built from this corpus with these options.
`--force` rebuilds it.

In [ ]:
!uv run zte-audit --root "{DATA_DIR}" --out "{DRIVE_ANALYSIS}"

## 6 · The encoder — where it stands after the 2026-08-15 sweep

### What has actually been measured

Real ZuCo, LOSO with `ZAB` held out, 160,804 words. The exp16 mechanisms were measured and two of the four were
falsified by their own matched ablations:

| measurement | value | reading |
| --- | --- | --- |
| `zte_encoder_v3` held-out Top-1 | 0.010 / 0.021 / 0.029 (seeds 42/43/44) | seed noise the size of the effect |
| `exp16_residual_off` held-out Top-1 | **0.0371**, eff-rank 0.289 | the residual subtracts the sentence code |
| `exp16_gallery_off` held-out Top-1 | 0.030 | the gallery CE hurts too |
| Effective-rank ratio (v3) | 0.06–0.09 | collapsed — no anti-collapse term guards the pooled tensor |
| Variance budget (v3) | 41.1% subject · 35.7% task · ~0% content | the space encodes who and which block, not what |
| Honest cell (train-fitted, length-stratified) | rank percentile **0.8775** [0.8695, 0.8855] | below the ±1 length oracle's 0.9525 — floor not cleared |
| Task probe | 0.918 vs 0.685 raw | the encoder *amplifies* the task register |

### The bit budget

Naming one of 700 sentences costs $\log_2 700 = 9.4512$ bits. Conditioning on word count leaves 4.3090, so length
alone supplies

$$
I(\text{identity};\, n_\text{words}) \;=\; H(\text{identity}) - H(\text{identity} \mid n_\text{words}) \;=\; 5.1422 \text{ bits}
$$

for free, and the encoder carries **0.6390** (train-fitted, full gallery). Fano's inequality prices an 80% pick over
700 sentences at ≥ 6.84 usable bits, so free-form identification stays out of reach — the readout has to fit the
channel instead.

### The readout that fits the channel

The tracked headline is now **menu capacity** (`zte-rebaseline` → `rebaseline.md` § Menu capacity): the largest
K-way closed set — distractors matched on task and *exact* word count — the embedding serves at ≥ 80% accuracy,
certified by CI lower bound and permutation p. At K = 2 the honest cell already sits near 88% on a never-seen
subject; every honest bit gained doubles the certified menu. The repair family that chases those bits is
`ablation/exp17_*` (§7c).


## 7 · Train the encoder

Every cell here is **resumable**. Re-run it verbatim after an interruption: finished runs are skipped, an
interrupted one continues from its last checkpoint, and each stage mirrors to Drive as it completes.

### 7a · Pick an arm

The dropdown reads `experiments/` live through `zte-colab arms`, so it always offers what is actually on disk and a
promoted config needs no edit here. Each entry is labelled by the config's own header comment. `experiments/archive/`
is the record of what failed and is never offered.

In [ ]:
ARMS = colab('arms', '--kind', 'encoder')
ENCODER_ARMS: dict[str, str] = {f'{a["tier"]} · {a["label"]}': a['path'] for a in ARMS['arms']}
HOLDOUTS: list[str] = ARMS['holdouts']

CONFIG: str = 'experiments/flagship/zte_encoder_v3.yaml'
HOLDOUT: str = 'ZAB'
SEED: int = 42


def _picker() -> None:
    """Offers the arm / held-out subject / seed as widgets, falling back to the assignments above off Colab."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print(f'ipywidgets unavailable — edit CONFIG / HOLDOUT / SEED above.\n  {CONFIG} · {HOLDOUT} · s{SEED}')
        return

    default = next((label for label, path in ENCODER_ARMS.items() if path == CONFIG), next(iter(ENCODER_ARMS)))
    arm = widgets.Dropdown(options=list(ENCODER_ARMS), value=default, description='arm:', layout={'width': '46em'})
    holdout = widgets.Dropdown(options=HOLDOUTS, value=HOLDOUT, description='hold out:')
    seed = widgets.IntText(value=SEED, description='seed:')
    out = widgets.Output()

    def _sync(_: object = None) -> None:
        globals().update(CONFIG=ENCODER_ARMS[arm.value], HOLDOUT=holdout.value, SEED=int(seed.value))
        with out:
            out.clear_output()
            print(f'{globals()["CONFIG"]}\n  hold out {globals()["HOLDOUT"]} · seed {globals()["SEED"]}')

    for control in (arm, holdout, seed):
        control.observe(_sync, names='value')
    _sync()
    display(widgets.VBox([widgets.HBox([holdout, seed]), arm, out]))


_picker()

### 7b · Run it

One config × one held-out subject. On a raw-conformer arm with 40 epochs this is a few hours; it checkpoints every
epoch and mirrors to Drive, so an interruption costs at most one epoch.

`--loso-holdout` names the held-out subject **and forces `train.split: by_subject_loso`** — the *closed-set* split,
in which all 700 stimuli are shared between train and val. That is the right split for an encoder: the question
there is whether the embedding reaches a brain it has never seen, and every stimulus being familiar is exactly what
isolates the subject as the variable.

It is the wrong split for a decoder, where a familiar stimulus means reciting the corpus rather than reading one —
so `zte-run` **refuses** the flag on a decoder or joint run whose config asked for a different split, and §8a names
the subject in the config instead.

`--seed` pins the run and `--data-cache` points at the staged bundle so nothing is re-prepared.

In [ ]:
RUN_NAME: str = f'{pathlib.Path(CONFIG).stem}_lo{HOLDOUT}_s{SEED}'
print('->', RUN_NAME)

!uv run zte-run \
  --config "{CONFIG}" \
  --root "{DATA_DIR}" \
  --name "{RUN_NAME}" \
  --out-root "{OUT_ROOT}" \
  --loso-holdout "{HOLDOUT}" \
  --seed {SEED} \
  --data-cache "{PREPARED_LOCAL}" \
  --drive-backup "{DRIVE_BACKUP}" \
  --spatial exact \
  --resume

### 7c · The ablations — one lever each

Every arm below is byte-identical to its named parent except for the single lever, so a difference in held-out rank
percentile is attributable to that lever and nothing else.

The exp16 family (parent `zte_encoder_v3.yaml`) has been measured — residual and gallery hurt, consensus helps:

| arm | lever | measured held-out Top-1 (s42) |
| --- | --- | --- |
| `exp16_residual_off` | `model.residual_coding` | **0.0371** vs 0.010 full — best measured encoder arm |
| `exp16_consensus_off` | the three `consensus_*` weights | 0.0057 — consensus is the mechanism that helps |
| `exp16_gallery_off` | `objective.gallery_weight` | 0.030 |
| `exp16_gallery_band_off` | `objective.gallery_length_band` | 0.027 |
| `exp16_length_projection_off` | `objective.length_projection` | 0.0086 — measurement-neutral |

The exp17 repair family (parent `exp17_base.yaml` = `exp16_residual_off` + `gallery_weight: 0`) is unmeasured; no
arm earns a claim without directional consistency across seeds 42/43/44:

| arm | lever | the question it answers |
| --- | --- | --- |
| `exp17_base` | `objective.gallery_weight` | Do the two measured-harmful mechanisms compound when both are removed? |
| `exp17_sent_vicreg` | sentence-slice VICReg | Does anti-collapse on the evaluated tensor hold rank without costing retrieval? |
| `exp17_task_blocked` | `objective.within_task_negatives` | Does removing the task subsidy convert register variance into content? |
| `exp17_align_train` | `dataset.raw_align_fit: train` | What is the fit-on-all alignment leak worth? |


In [ ]:
ABLATIONS: list[str] = [
    'experiments/ablation/exp16_residual_off.yaml',
    'experiments/ablation/exp16_consensus_off.yaml',
    'experiments/ablation/exp16_gallery_off.yaml',
    'experiments/ablation/exp16_gallery_band_off.yaml',
    'experiments/ablation/exp16_length_projection_off.yaml',
    'experiments/ablation/exp17_base.yaml',
    'experiments/ablation/exp17_sent_vicreg.yaml',
    'experiments/ablation/exp17_task_blocked.yaml',
    'experiments/ablation/exp17_align_train.yaml',
]

for config in ABLATIONS:
    name = f'{pathlib.Path(config).stem}_lo{HOLDOUT}_s{SEED}'
    print(f'\n=== {name} ' + '=' * 40)
    !uv run zte-run \
      --config "{config}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
      --loso-holdout "{HOLDOUT}" --seed {SEED} --data-cache "{PREPARED_LOCAL}" \
      --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume

mirror_to_drive()

### 7d · Multi-seed — the error bars a reviewer will ask for

A single seed is not a result on this corpus: the 2026-07-25 sweep moved between 2 and 9 hits in 700 across arms
whose *only* difference was noise, and an earlier re-run of one identical configuration gave 4 hits and then 2.

Three to five fixed seeds, reported as **mean ± sd**. The analysis section aggregates them automatically.

In [ ]:
SEEDS: tuple[int, ...] = (42, 43, 44)

for seed in SEEDS:
    name = f'{pathlib.Path(CONFIG).stem}_lo{HOLDOUT}_s{seed}'
    print(f'\n=== {name} ' + '=' * 40)
    !uv run zte-run \
      --config "{CONFIG}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
      --loso-holdout "{HOLDOUT}" --seed {seed} --data-cache "{PREPARED_LOCAL}" \
      --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume

mirror_to_drive()

### 7e · The full LOSO sweep — twelve strangers

One config against every subject in turn. This is the exhaustive *does it reach a brain it has never seen* trend and
the only number allowed to be called generalisation. **Multi-hour**, and resumable.

Read the result with `zte-loso-summary`, never with the per-fold pooled Top-1 in `INDEX.md`.

In [ ]:
!SPATIAL=exact DATA_CACHE="{PREPARED_LOCAL}" FULL_CFG="{CONFIG}" \
 DRIVE_BACKUP="{DRIVE_DIR}/loso" OUT_ROOT="{OUT_ROOT}/loso" \
 bash scripts/run_loso.sh "{DATA_DIR}"

!uv run zte-loso-summary --experiments "{OUT_ROOT}/loso" --out "{DRIVE_ANALYSIS}/LOSO.md"
mirror_to_drive(f'{OUT_ROOT}/loso', 'loso')

## 8 · The decoder — text out, on a short leash

The decoder reads a frozen LM through a small trainable bridge. Two mechanisms make it auditable rather than merely
fluent:

**The semantic rate ladder.** The conditioning vector passes through residual codebooks seeded by k-means on the
frozen *text* cloud, so the channel carries at most $\text{stages} \times \log_2(\text{codes})$ bits **by
construction**, and `bit_budget` reports how many actually arrived against the 9.45 needed. The bit budget stops
being an argument and becomes an instrument. Stage 0 is reserved for word count and the rest are penalised for
correlating with it, so `residual_mutual_information_bits` is the part the brain supplied.

**Word-synchronous lexical evidence.** A monotonic pointer walks the reading's words as the LM decodes — eye tracking
gives that alignment for free — nudging the LM's final hidden state, which through a linear frozen head *is* a
rank-limited logit bias. The pointer schedule is **content-free by construction**, which is what makes the
`length_only` control fair: it keeps the schedule and zeroes the content.

Two knobs to know when reading a decoder run. `decoder.rescore_pmi` scores each gallery candidate by its conditional
log-likelihood minus its null-prefix log-likelihood, cancelling the familiarity head start the train-fitted decoder
gives text the LM already finds likely. And the joint mode is now measurable: the `best.pt` monitor is stage-aware —
it resets at each curriculum boundary — so the decoder stage's best checkpoint is no longer masked by an earlier
stage's loss scale.

### The seven controls

Free-running generation is scored against every one of them, and the verdict fails if any is missing.

| control | what it removes | what it proves if the decode still wins |
| --- | --- | --- |
| `mean_prefix` | the reading, keeping the average | the answer is in *this* reading, not the cohort mean |
| `null_prefix` | the prefix entirely | the LM is not just being an LM |
| `phase` | the EEG's phase, keeping its spectrum | the signal is temporal structure, not band power |
| `noise` | the EEG, keeping the shape | Gaussian noise scoring the same means hallucinated priors |
| `shuffled_z` | the pairing, keeping the distribution | it is *this* brain state, not any brain state |
| `length_only` | the content, keeping the length schedule | it is lexical content, not word count |
| `mismatch` | the correspondence, pairing wrong readings | the alignment is doing the work |

Generation is evaluated **strictly autoregressively** — no ground-truth prefixes, greedy decode, recorded in the
artifact as `teacher_forced: false`. Teacher-forced perplexity is computed and quarantined as `*_DIAGNOSTIC`, never
read by the verdict.

In [ ]:
DECODER_BY_PATH: dict[str, dict[str, Any]] = {a['path']: a for a in colab('arms', '--kind', 'decoder')['arms']}
DECODER_ARMS: dict[str, str] = {f'{a["tier"]} · {a["label"]}': path for path, a in DECODER_BY_PATH.items()}
DECODER_CONFIG: str = 'experiments/flagship/decode_zte_v2.yaml'


def decoder_holdout() -> str:
    """The subject the chosen decoder config holds out -- it lives in the YAML, never on a CLI flag.

    `--loso-holdout` would force `by_subject_loso`, in which every gallery sentence is also a training
    sentence, so `zte-run` refuses it here and the config carries `train.loso_holdout_subject` instead.
    """
    arm = DECODER_BY_PATH[DECODER_CONFIG]
    if not arm['holdout']:
        raise ValueError(f'{DECODER_CONFIG} names no train.loso_holdout_subject — add one before training it.')

    return str(arm['holdout'])


def _decoder_picker() -> None:
    """Offers the decoder arm as a dropdown, falling back to the assignment above off Colab."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print(f'ipywidgets unavailable — edit DECODER_CONFIG above.\n  {DECODER_CONFIG}')
        return

    default = next((label for label, path in DECODER_ARMS.items() if path == DECODER_CONFIG), next(iter(DECODER_ARMS)))
    arm = widgets.Dropdown(options=list(DECODER_ARMS), value=default, description='decoder:', layout={'width': '58em'})
    out = widgets.Output()

    def _sync(_: object = None) -> None:
        globals()['DECODER_CONFIG'] = DECODER_ARMS[arm.value]
        chosen = DECODER_BY_PATH[globals()['DECODER_CONFIG']]
        with out:
            out.clear_output()
            print(globals()['DECODER_CONFIG'])
            print(f'  split {chosen["split"]} · holds out {chosen["holdout"]}')

    arm.observe(_sync, names='value')
    _sync()
    display(widgets.VBox([arm, out]))


_decoder_picker()

### 8a · Train the bridge over the frozen encoder

The decoder is trained over a **frozen** encoder checkpoint, so the number it produces is attributable to the bridge
and not to a quietly-retrained encoder. Point `--encoder-ckpt` at the arm you trained in §7.

**There is no `--loso-holdout` on this cell, and that is the whole point.** The flag forces `by_subject_loso`, which
shares all 700 stimuli between train and val: every gallery sentence would also be a training sentence, and the
generation verdict's `honest_split` clause could never pass — nothing such a run measured could ever be a headline.
`zte-run` refuses it here.

The held-out subject is named in the config instead, as `train.loso_holdout_subject`, beside
`train.split: by_subject_and_stimulus` — which holds out the subject **and** their sentences. That is what finally
makes the honest-split clause reachable. The cell reads the subject back out of the chosen config, so a run
directory can never be labelled with a subject the config did not actually hold out.

In [ ]:
# Drive first: on a fresh runtime the encoder you trained last session is on Drive and not on this disk.
ENCODER_CKPT: str = resolve_ckpt(RUN_NAME)
DECODER_HOLDOUT: str = decoder_holdout()
DECODER_NAME: str = f'{pathlib.Path(DECODER_CONFIG).stem}_lo{DECODER_HOLDOUT}_s{SEED}'
print(f'decoder -> {DECODER_NAME}   (holds out {DECODER_HOLDOUT}, named by {DECODER_CONFIG})')

!uv run zte-run \
  --config "{DECODER_CONFIG}" --root "{DATA_DIR}" --name "{DECODER_NAME}" --out-root "{OUT_ROOT}" \
  --encoder-ckpt "{ENCODER_CKPT}" --seed {SEED} --data-cache "{PREPARED_LOCAL}" \
  --drive-backup "{DRIVE_BACKUP}" --resume

### 8b · Decode with every control, over several seeds

`zte-decode` runs the headline decode and all seven controls through **one** code path, so the comparison is paired
rather than approximate. `--seeds` repeats the whole thing and reports the spread of the worst-control delta, which
is the quantity the verdict actually gates on.

`--capacity` adds the menu-capacity certification off the *same* gallery pass — no extra LM forward, no second
model — and writes it to `capacity.json`. §8e reads it.

It writes `generation.json`, `generation.jsonl` and `capacity.json` into `--out`. **Sections 8d and 8e read exactly
those files** — nothing below re-decodes or re-certifies a subset, because a number the verdict did not gate on is
not the number to look at.

Once those files exist, re-running this cell is free: the decode is skipped when they were written from this exact
checkpoint with these exact options, and redone the moment any of that moves. `--force` redoes it deliberately.

In [ ]:
!uv run zte-decode \
  --ckpt "{resolve_ckpt(DECODER_NAME)}" \
  --root "{DATA_DIR}" --split test \
  --controls mean_prefix,null_prefix,phase,noise,shuffled_z,length_only,mismatch \
  --seeds 42,43,44 \
  --within-task \
  --capacity \
  --out "{DRIVE_ANALYSIS}/decode_{DECODER_NAME}"

### 8c · The length audit — read this before any decoder number

`zte-rebaseline` measures how much of a checkpoint's retrieval a length-only oracle reproduces, with no retraining.
It gates nothing; it tells you which column of the report to trust.

On the current best decoder, length-stratified rescoring rank percentile was **0.4349** — below the 0.5 chance line.
That is the whole reason the reserved length stage and the `length_only` control exist.

It skips itself the way §8b does, and re-audits as soon as the checkpoint has trained further.

In [ ]:
!uv run zte-rebaseline \
  --ckpt "{resolve_ckpt(DECODER_NAME)}" \
  --root "{DATA_DIR}" \
  --out "{DRIVE_ANALYSIS}/rebaseline_{DECODER_NAME}.json"

### 8d · Read the decodes — target beside hypothesis beside every control

`zte-colab readings` reads the artifacts §8b wrote: for each held-out reading, the sentence the person read, the
sentence the decoder wrote from their EEG, and the same decode from each brain-independent condition. It decodes
nothing itself — §8b already scored every reading against every pre-registered control, and re-running a subset here
would show numbers the verdict does not gate on.

Read it in this order, and only in this order:

1. **The verdict first.** `generation_above_controls` ANDs over five clauses, and a control that could not run
   *fails* its clause rather than vanishing from the gate.
2. **Then the controls.** A frozen LM reaches ROUGE-1 in the 0.10–0.18 range against *any* English reference from
   function words alone, so an absolute score is not evidence of anything. `null_prefix` is the floor the language
   model gets for free.
3. **Then the paired delta.** Hypothesis minus control, on the same reading — that is the only quantity that carries
   information about the brain.
4. **Then remember the arithmetic.** The encoder supplies ~1.5 bits and a 19.6-word sentence needs ~190. An honest
   null here is the expected result, and reporting it plainly is the finding.

`length_only` is the control that matters most for this project. It keeps the pointer schedule — so it still gets the
word count, ZuCo's free 5.14 bits — and destroys only *what each word was*. A hypothesis that beats it beat it on
lexical content and on nothing else.

In [ ]:
import pandas as pd
from IPython.display import display

DECODE_OUT: str = f'{DRIVE_ANALYSIS}/decode_{DECODER_NAME}'
READINGS = colab('readings', '--from', DECODE_OUT, '--rows', '12')

src, verdict = READINGS['source'], READINGS['verdict']
print(f'{src["run_name"]}   ·   {src["split"]} cell of {src["split_strategy"]}')
print(f'{src["n_total"]} held-out readings, {src["n_scored"]} scored, {src["n_shown"]} shown below')
print(f'primary metric: {READINGS["primary_metric"]}\n')

print(f'generation_above_controls : {verdict["above_controls"]}')
for clause, passed in verdict['clauses'].items():
    print(f'  {"PASS" if passed else "FAIL"}   {clause}')
if verdict.get('controls_absent'):
    print(f'  controls that never ran (each one fails its clause): {", ".join(verdict["controls_absent"])}')
if not READINGS['applicable']:
    print(f'  not scored: {READINGS["reason"]}')

# Absolute scores are unreadable alone; the paired delta beneath them is the quantity that carries the brain.
display(pd.DataFrame(READINGS['summary']).T.round(4))
display(pd.DataFrame(READINGS['deltas']).T)

In [ ]:
# One reading at a time: the sentence read, the sentence written, and each control's attempt at the same row.
for reading in READINGS['readings'][:4]:
    print('=' * 108)
    print(
        f'{reading["subject"]} · {reading["task"]} · {reading["n_words"]} words · KL {reading["prefix_influence_kl"]}'
    )
    print(f'  TARGET       : {reading["target"]}')
    for condition in reading['conditions']:
        scores = '  '.join(f'{name}={value:.4f}' for name, value in condition['scores'].items())
        print(f'  {condition["name"]:<13}: {str(condition["text"])[:96]!r}   {scores}')

### 8e · Menu capacity — the number this project can actually prove

**Free generation is a measured null, and it will stay one.** The conditioning channel carries on the order of
**1.5 bits**; a 19.6-word ZuCo sentence needs about **190**. That is under 1% of what writing a sentence from
scratch requires, so §8d's honest null is arithmetic, not a training failure. Reporting it plainly is the finding.

The winnable question — and the clinically meaningful one, because a communication device offers choices — is
**how many alternatives can this decoder tell apart?** Given the held-out reading and K candidate sentences, does
it score the sentence the person actually read above every distractor? That is a menu, and a menu is what a
locked-in user would actually be handed.

Four things make the number honest:

- **The distractors are not easy.** Every candidate shares the true sentence's **exact word count** and its
  **task**, so a hit cannot come from ZuCo's free 5.14 bits of length, nor from the register difference between
  normal and task-specific reading.
- **Chance is exactly $1/K$** — the closed-form expectation over uniformly drawn distractors, not a sampled
  estimate — and **ties lose**. A decoder that separates nothing scores **0.0**, not chance. A number near chance
  is therefore already a live signal, and a zero is a dead one.
- **Certification is contiguous and conservative.** A size K counts only if it and *every smaller size swept*
  clear all seven clauses, and only if the same holds on the common subset of queries scoreable at every size —
  otherwise a rising tail is just a shrinking, surviving subpopulation.
- **K = 32 and K = 64 are usually unreachable.** An exact word-count pool holds a median of ~8 candidates on a
  300-sentence gallery and ~18 on a 700-sentence one. Those sizes come back as *unreachable* — a pool that cannot
  be built — never as a decoder that failed. The cell prints them under their own label.

**The only line that matters is the model-vs-`length_only` gap**, measured **paired, per query**, on the same
pools. `length_only` is the same bridge, the same frozen LM, the same scaffold and the same length normalisation,
conditioned on a length-matched training prefix instead of this reading's EEG. Everything the word count can buy,
it also buys. The gap is what is left, and it is the only part attributable to the brain.

> `certified_k` coming back as **—** is the expected first result on real ZuCo, and it is a result. The cell names
> the clause that stopped it. An em dash is never a zero and never a blank.

In [ ]:
CAPACITY = colab('capacity', '--from', DECODE_OUT)
src, sel, bits = CAPACITY['source'], CAPACITY['selected'], CAPACITY['bits']


def dash(value: Any, digits: int = 4) -> str:
    """A number, or an em dash where there is none -- a blank or a zero would read as a measurement."""
    if value is None:
        return '—'

    return f'{value:.{digits}f}' if isinstance(value, float) else str(value)


print(f'{src["run_name"]} · holds out {src["holdout"]} · {src["n_queries"]} queries · {src["n_gallery"]} gallery')
print(f'{CAPACITY["readout"]} · {CAPACITY["tie_policy"]} · score {sel["score"]} · pool {sel["flavor"]}')
print(f'split {CAPACITY["split_strategy"]} / {CAPACITY["split_cell"]}')
if sel['substituted']:
    print(f'!  {sel["requested_score"]}/{sel["requested_flavor"]} was never swept — the pool read is the one above')
if sel['gamed']:
    print('!  a length oracle wins this pool — it is length-gamed and certifies nothing')

print(f'\nCERTIFIED K : {dash(CAPACITY["certified_k"])}      bits : {dash(bits["bits_certified"])}')
for clause, passed in CAPACITY['clauses'].items():
    print(f'  {"PASS" if passed else "FAIL"}   {clause}')
print(f'  {CAPACITY["verdict"]["reason"]}')
print(f'  menu sizes no pool could fill: {CAPACITY["ks"]["unreachable"] or "none"}')
if CAPACITY['pooled']:
    print(f'  pooled over seeds: {CAPACITY["pooled"]["reason"]}')

In [ ]:
import pandas as pd
from IPython.display import display

# Every swept size, unreachable ones kept in place: a size that vanishes reads as a size that was never tried.
per_k = pd.DataFrame(CAPACITY['per_k'])
per_k['pool'] = per_k['reachable'].map({True: 'filled', False: 'UNREACHABLE'})
per_k['failed'] = per_k['failed_clauses'].map(lambda names: ', '.join(names) or '—')
columns = ['k', 'pool', 'n_queries', 'chance', 'accuracy', 'ci_lo', 'ci_hi', 'perm_p', 'certified', 'failed']
display(per_k[columns].round(4))

# The paired gap, per query, on identical pools — the only quantity here that carries the brain.
display(pd.DataFrame(CAPACITY['paired']).round(4))

# Bits are read against the 4.3090 that survive knowing word count, never against the full 9.4512.
display(pd.DataFrame([bits]).T.rename(columns={0: 'value'}))

In [ ]:
import plotly.graph_objects as go

filled = [row for row in CAPACITY['per_k'] if row['reachable']]
sizes = [row['k'] for row in filled]
axis = {'type': 'log', 'tickvals': sizes, 'title': 'menu size K'}

ladder = go.Figure()
ladder.add_scatter(x=sizes, y=[row['chance'] for row in filled], name='chance = 1/K', line={'dash': 'dot'})
for arm in sorted({row['arm'] for row in CAPACITY['arms']}):
    rows = [row for row in CAPACITY['arms'] if row['arm'] == arm and row['k'] in sizes]
    bars = {
        'type': 'data',
        'symmetric': False,
        'array': [row['ci_hi'] - row['accuracy'] for row in rows],
        'arrayminus': [row['accuracy'] - row['ci_lo'] for row in rows],
    }
    ladder.add_scatter(x=[row['k'] for row in rows], y=[row['accuracy'] for row in rows], name=arm, error_y=bars)
ladder.update_layout(title='Menu accuracy, model against every control', xaxis=axis, yaxis_title='accuracy')
ladder.show()

gap = go.Figure()
for control in sorted({row['control'] for row in CAPACITY['paired']}):
    rows = [row for row in CAPACITY['paired'] if row['control'] == control and row['k'] in sizes]
    bars = {
        'type': 'data',
        'symmetric': False,
        'array': [row['ci_hi'] - row['delta'] for row in rows],
        'arrayminus': [row['delta'] - row['ci_lo'] for row in rows],
    }
    gap.add_scatter(x=[row['k'] for row in rows], y=[row['delta'] for row in rows], name=control, error_y=bars)
gap.add_hline(y=0.0, line_dash='dot')
gap.update_layout(title='Paired gap: model minus control, same query, same pool', xaxis=axis, yaxis_title='Δ')
gap.show()

### 8f · The decode studio — watch it happen

`zte-studio` decodes a handful of held-out readings **with a full per-step trace** and writes one self-contained
interactive page. It is the same `generate_from_prefix` call the evaluation makes — the trace is a sink the loop
writes to and nothing else, so the page cannot show a decode the evaluation did not make.

What is on it, and the real quantity behind each part:

| panel | what it is actually reading |
| --- | --- |
| **Scalp field** (2D cap · draggable 3D head) | per-word band power interpolated across the montage, at the word the pointer is on |
| **Target sentence** | the pointer's Gaussian window over the reading's words, at the current decoding step |
| **Decoded text** | tokens revealed as they were emitted, shaded by probability; click one to jump to that step |
| **Alternatives** | the top-8 next-token distribution at that step — what it nearly said |
| **Evidence KL** | the same hidden state with and without the word-synchronous nudge: how hard the brain pushed on *this* token |
| **Pointer walk** | the full (step × word) attention matrix |
| **Rate-ladder codes** | which codebook entry each stage selected for this reading |

Space plays and pauses, the arrow keys step, and the scrub bar seeks. The band buttons switch which rhythm the scalp
map shows — theta and gamma are the lexical-semantic pair, alpha and beta the attentional one.

**The scalp colour scale is relative within one reading**, so two readings whose maps look alike are not therefore
alike in microvolts. The page says so on itself.

> **It is an inspection tool, not an audit.** A handful of readings chosen to look at is an anecdote; the verdict
> needs the paired delta over every held-out reading, its bootstrap interval and the permutation null. The page
> carries that warning in its own banner so a screenshot cannot be mistaken for a result.

In [ ]:
from IPython.display import IFrame

DECODE_CKPT: str = resolve_ckpt(DECODER_NAME)
STUDIO: str = f'{DRIVE_ANALYSIS}/STUDIO_{DECODER_NAME}.html'

!uv run zte-studio \
  --ckpt "{DECODE_CKPT}" \
  --root "{DATA_DIR}" \
  --split test \
  --rows 8 \
  --controls null_prefix,length_only,mismatch \
  --montage res/montage_gsn105.csv \
  --out "{STUDIO}"

# Copy to the VM disk before embedding: an iframe reading straight off the Drive mount is slow enough to look broken.
local_studio = pathlib.Path('res/analysis/STUDIO.html')
local_studio.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(STUDIO, local_studio)
print(f'{local_studio}  ({local_studio.stat().st_size / 1e6:.1f} MB)  ·  also on Drive at {STUDIO}')
display(IFrame(src=str(local_studio), width='100%', height=900))

## 9 · The whole study in one command

`scripts/run_zte_study.sh` runs every stage in order and is safe to re-run after any interruption — each stage skips
work already on disk and mirrors to Drive as it finishes.

| stage | what it does |
| --- | --- |
| `audit` | the model-free confound report |
| `encoder` | the flagship encoder arms |
| `loso` | the full twelve-fold sweep |
| `decoder` | the decoder arms and their controls |
| `ablation` | one-lever studies |
| `rebaseline` | the length-oracle audit |
| `analysis` | the dashboard and its tidy tables |

Set `STAGES` to a subset to run only part of it. The dashboard lands in `$OUT_ROOT/analysis` and is mirrored to
`$DRIVE_BACKUP/analysis`, so it survives the VM either way. **This is the multi-hour cell** — start it, and come
back.

In [ ]:
!SEEDS="42 43 44" \
 STAGES="audit encoder loso decoder ablation rebaseline analysis" \
 SPATIAL=exact \
 DATA_CACHE="{PREPARED_LOCAL}" \
 OUT_ROOT="{OUT_ROOT}" \
 DRIVE_BACKUP="{DRIVE_BACKUP}" \
 bash scripts/run_zte_study.sh "{DATA_DIR}"

mirror_to_drive()

## 10 · Analysis — everything you have ever run, visually

`zte-analyze` walks any number of run trees, collects every artifact into tidy frames, and writes one **self-contained
offline HTML page** plus the CSVs behind it. Plotly is inlined, so the page opens from a Drive mirror on a machine
with no network.

It reads across **every session folder on Drive**, not just today's, so the picture is cumulative.

In [ ]:
ANALYSIS_ROOTS: list[str] = [*every_session(), LOCAL_RUNS]
MONTAGE: list[str] = ['--montage', 'res/montage_gsn105.csv'] if os.path.isfile('res/montage_gsn105.csv') else []
ROOTS: str = ' '.join(f'"{root}"' for root in ANALYSIS_ROOTS)
MONTAGE_FLAG: str = ' '.join(MONTAGE)

print('reading:')
for root in ANALYSIS_ROOTS:
    print('  ', root)

!uv run zte-analyze --experiments {ROOTS} --out "{DRIVE_ANALYSIS}" {MONTAGE_FLAG}

### 10a · The tables that carry the argument

`zte-analyze` wrote every tidy frame behind the page as CSV, so the notebook reads them with Colab's own pandas and
the numbers in the cell below are byte-identical to the ones in the page. Every cell is **mean ± sd across seeds**,
and `n_seeds` travels with it so a single-run row cannot be mistaken for a stable one.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', 60, 'display.width', 200, 'display.float_format', '{:.4f}'.format)
TABLES = pathlib.Path(DRIVE_ANALYSIS) / 'tables'


def table(name: str) -> pd.DataFrame:
    """One tidy frame `zte-analyze` wrote, or an empty frame when nothing was collected for that view."""
    path = TABLES / f'{name}.csv'

    return pd.read_csv(path) if path.is_file() else pd.DataFrame()


KEEP: list[str] = [
    'arm',
    'n_seeds',
    'held_out_rank_percentile_mean',
    'held_out_rank_percentile_sd',
    'held_out_top1_mean',
    'stratified_rank_percentile_mean',
    'effective_rank_ratio_mean',
]
seeds = table('multi_seed')
if seeds.empty:
    print(f'no tables under {TABLES} yet — run the cell above first')
else:
    display(seeds[[c for c in KEEP if c in seeds]].sort_values('held_out_rank_percentile_mean', ascending=False))

In [ ]:
for name, frame in (
    ('Per-lever ablation', table('feature_ablation')),
    ('LOSO, per held-out subject', table('loso')),
    ('Within-task pools (SR / NR)', table('within_task')),
):
    print(f'\n### {name}')
    display(frame if not frame.empty else 'nothing collected for this view yet')

### 10b · The panels, inline and interactive

`zte-colab panels` draws the same charts the page carries and writes each one as Plotly figure JSON, which this
kernel renders with Colab's own plotly. One list of panels, one set of captions, drawn once — the notebook cannot
show a chart the page does not.

A panel with nothing behind it is **named rather than dropped**, because a chart that silently vanishes reads like a
chart that had nothing to say.

- **Pick a headline** — every metric behind one dropdown.
- **Did the mechanism engage?** — per-epoch training curves. A consensus term that never fired or a gallery accuracy
  pinned at chance is visible *only* here; the final metrics cannot tell "did nothing" from "was never switched on".
- **Who vs what** — the subject probe against the content probe, sized by effective rank, because a probe that falls
  on both axes has collapsed rather than become invariant.
- **Length leakage** — before and after the projection.
- **Bit budget** — the 9.45 bits, and who supplies them.
- **Electrode map** — the scalp geometry the encoder actually reads.

In [ ]:
import plotly.io as pio

PANELS = colab('panels', '--experiments', *ANALYSIS_ROOTS, '--out', f'{DRIVE_ANALYSIS}/panels', *MONTAGE)
collected = PANELS['study']
print(f'{collected["runs"]} run(s) · {collected["folds"]} fold row(s) · {collected["generations"]} decode row(s)')
if collected['synthetic_runs']:
    print(f'⚠  {collected["synthetic_runs"]} of them are SYNTHETIC — wiring checks, and never results.')

for panel in PANELS['panels']:
    print(f'\n#### {panel["name"]} — {panel["caption"]}')
    pio.read_json(panel['path']).show()

if PANELS['empty']:
    print('\nno data collected yet for: ' + ', '.join(PANELS['empty']))

### 10c · Drill into one run

The dropdown lists every run the analysis found. Selecting one prints its honest headline block and shows its saved
figures, so a suspicious number can be chased to the run that produced it without leaving the notebook.

In [ ]:
def _run_explorer() -> None:
    """A dropdown over every collected run that prints its honest headline block and displays its figures."""
    found = find_runs(*ANALYSIS_ROOTS, headline=True)
    if not found:
        print('no runs found yet')
        return

    try:
        import ipywidgets as widgets
        from IPython.display import Image, display
    except ImportError:
        print('ipywidgets unavailable — runs found:', ', '.join(r['name'] for r in found))
        return

    by_name = {r['name']: r for r in found}
    picker = widgets.Dropdown(options=list(by_name), description='run:', layout={'width': '46em'})
    out = widgets.Output()

    def _show(_: object = None) -> None:
        run = by_name[picker.value]
        head = run['headline']
        with out:
            out.clear_output()
            print(f'{run["name"]}   ({run["source"]}{", " + run["session"] if run["session"] else ""})')
            if run['synthetic']:
                print('  SYNTHETIC — a wiring check on 372 words, never a result')
            if head is None:
                print(f'  not evaluated yet: {run["path"]}')
                return

            print(f'  held out                 : {head["holdout_subject"]}')
            print(f'  held-out rank percentile : {head["held_out_rank_percentile"]}')
            print(
                f'  held-out Top-1           : {head["held_out_top1"]} of {head["held_out_n_queries"]} '
                f'(chance {head["held_out_chance"]})'
            )
            print(f'  length-stratified rank   : {head["stratified_rank_percentile"]}')
            print(f'  effective rank ratio     : {head["effective_rank_ratio"]}')
            print(f'  postprocess fit          : {head["postprocess_fit"]}')
            print(f'  verdict                  : {json.dumps(head["verdict"] or {}, indent=2)[:600]}')
            for figure in run['figures'][:8]:
                display(Image(filename=figure, width=620))

    picker.observe(_show, names='value')
    _show()
    display(widgets.VBox([picker, out]))


_run_explorer()

### 10d · The full page

A few megabytes with Plotly inlined, so the frame below can be sluggish — the file itself is on Drive and opens
fastest in its own tab. It is also the artifact to share: no server, no network, no dependencies.

In [ ]:
from IPython.display import HTML, IFrame

page = pathlib.Path(DRIVE_ANALYSIS) / 'ANALYSIS.html'
print(page, '·', f'{page.stat().st_size / 1e6:.1f} MB' if page.is_file() else 'not built yet')

summary = pathlib.Path(DRIVE_ANALYSIS) / 'ANALYSIS.md'
if summary.is_file():
    print(summary.read_text()[:2000])

local_copy = pathlib.Path('res/analysis/ANALYSIS.html')
if page.is_file():
    local_copy.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(page, local_copy)
    display(IFrame(src=str(local_copy), width='100%', height=760))
else:
    display(HTML('<p>Run §10 first.</p>'))

## 11 · Persist, resume, continue offline

Three levels, and they answer different questions.

- **Mirror** — a browsable copy of the runs on Drive. Already automatic after each stage; call it again any time.
- **Archive** — a provenance-stamped zip of the best checkpoints, so a result can be reproduced later.
- **Snapshot** — the entire working state, *including the prepared data cache*, in one file. Download it and keep
  working on your own machine with no GPU time: `zte-pack unpack <zip> --dest res`.

If the runtime resets, `restore_from_drive()` pulls the session back to the VM and every training cell picks up from
its last checkpoint. The cells after training resume too, and differently: `zte-audit`, `zte-decode`,
`zte-rebaseline` and `zte-parallax transfer` each record what their artifacts were built from, skip when nothing has
moved, and rebuild as soon as the checkpoint, the data or an option changes. Re-run the notebook top to bottom and
only the unfinished work costs anything; `--force` on any one of them redoes it deliberately.

In [ ]:
import datetime


def archive_to_drive(note: str | None = None) -> None:
    """Provenance-stamped zip of the best checkpoints, skipping synthetic smoke runs."""
    stamp = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_{RUN_DATE}_{stamp}.zip'
    command = ['uv', 'run', 'zte-pack', 'zip', '--all', '--best-only', '--skip-synthetic', '--out', out]
    subprocess.run([*command, *(['--note', note] if note else [])], check=False)
    print('archive ->', out)


def snapshot_to_drive(note: str | None = None) -> str:
    """Everything -- runs, cache, benchmark, explorer -- in one zip, so a local session never re-prepares data."""
    stamp = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_snapshot_{RUN_DATE}_{stamp}.zip'
    command = ['uv', 'run', 'zte-pack', 'snapshot', '--skip-synthetic', '--out', out]
    subprocess.run([*command, *(['--note', note] if note else [])], check=False)
    print('snapshot ->', out)

    return out


mirror_to_drive()
archive_to_drive(note=f'{RUN_DATE} encoder v3 + decoder v2')

## 12 · Running it on your own machine

Nothing here is Colab-specific below the Drive cells. On a workstation with a real GPU the same twenty entry points
do the same work:

```sh
uv sync                                              # 'all' and 'dev' are default groups
uv run zte-prepare  --root <data> --configs          # build the feature bundles once

uv run zte-run      --config experiments/flagship/zte_encoder_v3.yaml \
                    --root <data> --loso-holdout ZAB --seed 42 --resume
uv run zte-audit    --root <data>                    # the model-free confound report
uv run zte-decode   --ckpt <ckpt> --root <data> --split test --seeds 42,43,44
uv run zte-rebaseline --ckpt <ckpt> --root <data>    # how much of a number is sentence length
uv run zte-loso-summary --experiments res/experiments/loso
uv run zte-analyze  --experiments res/experiments --out res/analysis
uv run zte-ablate   generate --config <cfg> --knob objective.gallery_length_band --values 0,1,2,4

SEEDS='42 43 44' bash scripts/run_zte_study.sh <data>   # the whole study, resumable
```

`zte-colab` is the twentieth, and it is what every cell above goes through: one subcommand per question — `env`,
`session`, `runs`, `arms`, `readings`, `panels`, `mirror` — each printing a single JSON object on stdout with its
logs on stderr. Run `uv run zte-colab --help` to see them. It is useful outside a notebook too, wherever a payload
is easier to read than an import.

**Device support.** CPU, CUDA, MPS and XLA are all live through `zte.device.resolve_device`. MPS has a ~30 GiB
ceiling and misses operators the CUDA path has — `raw_conformer` at batch 64 will not fit — so a local Apple machine
is for inference, analysis and the synthetic smoke path, not for training the raw arms.

**Before reporting anything**, run the gates the repository is held to:

```sh
uv run ruff format . && uv run ruff check . && uv run mypy src tests && uv run pytest
```

---

## 13 · Housekeeping

The VM disk fills up faster than you expect, mostly with checkpoints you have already mirrored.

In [ ]:
!uv run zte-pack list


def remove_locally(*names: str) -> None:
    """Delete local run directories or res/ subpaths to free disk. Never touches Drive."""
    for name in names:
        path = pathlib.Path(name)
        if not path.exists():
            path = pathlib.Path(LOCAL_RUNS) / name
        if path.exists():
            shutil.rmtree(path)
            print('removed', path)
        else:
            print('not found:', name)


# remove_locally('res/experiments/smoke_run', 'res/cache')
show_resources()